# Pac-Man in Gymnasium (Google Colab)

This notebook:
1. Installs Gymnasium + Atari (ALE) dependencies and the Pac-Man ROM
2. Creates the `ALE/MsPacman-v5` environment
3. Prints the **observation (state) space**, **action space**, and **rewards**
4. Renders frames and plays a short episode with random actions
5. Displays the episode as an animation, inline in Colab

> Tip: In Colab, go to **Runtime > Change runtime type** and select a GPU/CPU as needed (Atari + random actions work fine on CPU).

## 1. Install dependencies

`ale-py` provides the Arcade Learning Environment (Atari) backend, and `AutoROM` fetches the ROMs (including Ms. Pac-Man) under the Atari ROM license.

In [ ]:
!pip install -q "gymnasium[atari]" "gymnasium[accept-rom-license]" ale-py "autorom[accept-rom-license]"
!AutoROM --accept-license

## 2. Imports

In [ ]:
import gymnasium as gym
import ale_py

# Register ALE environments with Gymnasium
gym.register_envs(ale_py)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

## 3. Create the environment and print its specification

We use `render_mode="rgb_array"` because Colab has no display for `"human"` rendering — we'll instead grab RGB frames and show them with matplotlib.

In [ ]:
env = gym.make("ALE/MsPacman-v5", render_mode="rgb_array")

print("="*60)
print("ENVIRONMENT SPEC")
print("="*60)
print("Environment ID     :", env.spec.id)
print()
print("Observation space  :", env.observation_space)
print("Observation shape  :", env.observation_space.shape)
print("Observation dtype  :", env.observation_space.dtype)
print()
print("Action space       :", env.action_space)
print("Number of actions  :", env.action_space.n)
print("Action meanings    :", env.unwrapped.get_action_meanings())
print()
print("Reward range       :", env.reward_range if hasattr(env, 'reward_range') else 'n/a')
print("Max episode steps  :", env.spec.max_episode_steps)

## 4. Reset the environment and inspect the initial state

In [ ]:
obs, info = env.reset(seed=42)

print("Initial observation shape :", obs.shape)
print("Initial observation dtype :", obs.dtype)
print("Initial observation range :", obs.min(), "-", obs.max())
print("Info dict returned by reset():", info)

## 5. Render the first frame

In [ ]:
frame = env.render()

plt.figure(figsize=(4, 5))
plt.imshow(frame)
plt.axis("off")
plt.title("Initial Pac-Man frame")
plt.show()

## 6. Play an episode with random actions, printing state/action/reward

Each step we print the action taken, the reward received, and running total. We also store every rendered frame so we can animate the episode afterward.

In [ ]:
obs, info = env.reset(seed=0)
frames = [env.render()]

total_reward = 0.0
num_steps = 300

for step in range(num_steps):
    action = env.action_space.sample()  # random policy
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    frames.append(env.render())

    if reward != 0:
        print(f"Step {step:4d} | action={action} ({env.unwrapped.get_action_meanings()[action]:>10}) "
              f"| reward={reward:+.1f} | total_reward={total_reward:.1f} "
              f"| obs.shape={obs.shape}")

    if terminated or truncated:
        print(f"\nEpisode ended at step {step}. terminated={terminated}, truncated={truncated}")
        break

print(f"\nFinal total reward after {step+1} steps: {total_reward}")
print(f"Collected {len(frames)} frames for animation.")

## 7. Animate the episode inline (Colab-friendly)

Colab can't open a live game window, so instead we build a matplotlib animation from the saved frames and render it as HTML/JS directly in the notebook output.

In [ ]:
def display_frames_as_animation(frames, fps=30, stride=2):
    """Turn a list of RGB frames into an inline HTML5/JS animation.
    `stride` subsamples frames to keep the animation lightweight in Colab.
    """
    frames = frames[::stride]
    fig = plt.figure(figsize=(frames[0].shape[1] / 72, frames[0].shape[0] / 72), dpi=72)
    plt.axis("off")
    patch = plt.imshow(frames[0])

    def animate(i):
        patch.set_data(frames[i])
        return [patch]

    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames), interval=1000 / fps
    )
    plt.close(fig)
    return anim

anim = display_frames_as_animation(frames)
HTML(anim.to_jshtml())

## 8. Clean up

In [ ]:
env.close()
print("Environment closed.")

## Notes

- **State/observation space**: by default `ALE/MsPacman-v5` returns raw RGB frames of shape `(210, 160, 3)`, `uint8`. You can switch to a grayscale/downscaled representation with `AtariPreprocessing`, or to RAM state with `ALE/MsPacman-ram-v5` (observation space `Box(0, 255, (128,), uint8)`).
- **Action space**: `Discrete(9)` — the 9 joystick directions/no-op available for Ms. Pac-Man (`NOOP, UP, RIGHT, LEFT, DOWN, UPRIGHT, UPLEFT, DOWNRIGHT, DOWNLEFT`).
- **Reward**: score increase from eating pellets/fruit/ghosts on that step (0 most steps, positive on pickups).
- Replace the random policy (`env.action_space.sample()`) with a trained agent (DQN, PPO, etc.) to see real gameplay instead of random moves.